In [1]:
import requests
import pandas as pd
from util import find_teams

# A session makes sure our python script remembers things we did previously, like getting a cookie
# Without a session, we wouldn't retain the cookie we got from the front page and then we'd turn up to the API empty-handed
session = requests.Session()
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",  # this is a standard web browser fingerprint
    "X-Requested-With": "XMLHttpRequest",  # makes it look like it's the website's JavaScript itself fetching the data to populate the page
}
# We don't need anything from the front page but we want to look like a normal human who first went to the
# front page, got a cookie, then our browser showed the cookie to the API and called it
print("Step 1: Establishing Session & Pulling Payload...")
session.get("https://understat.com/league/EPL", headers=headers)
# This is getting the actual data from the hidden football data API
response = session.get("https://understat.com/getLeagueData/EPL/2025", headers=headers)

if response.status_code == 200:  # if we get in successfully:
    print("Yep, we're in!")
    raw_data = response.json()
    print("Step 2: Executing Recursive Search to bypass the wrappers...")
    extracted_teams = find_teams(raw_data)
    print("Got the team data!")
else:
    RuntimeError("We didn't make it in.")

Step 1: Establishing Session & Pulling Payload...
Yep, we're in!
Step 2: Executing Recursive Search to bypass the wrappers...
Got the team data!


In [2]:
from datetime import datetime

# from util import get_weighted_stats

# # Define our model parameters
target_date = datetime.now()  # We are making predictions relative to today
half_life_days = 90.0  # You can backtest this number later to find the optimal setting

# league_data = []
# for team in extracted_teams:
#     team_name = team["title"]

#     # Find home and away match data for each team
#     home_matches = [match for match in team["history"] if match["h_a"] == "h"]
#     away_matches = [match for match in team["history"] if match["h_a"] == "a"]

#     # Find home/away xG/xGA for each team
#     # Note that Understat sometimes stores xG/xGA as strings, so we convert to float to be safe
#     home_xg_per_game, home_xga_per_game, home_weight = get_weighted_stats(
#         home_matches, target_date, half_life_days
#     )
#     away_xg_per_game, away_xga_per_game, away_weight = get_weighted_stats(
#         away_matches, target_date, half_life_days
#     )

#     league_data.append(
#         {
#             "Team": team_name,
#             "Home_Weight": round(home_weight, 2),
#             "Away_Weight": round(away_weight, 2),
#             "Home_xG_per_game": round(home_xg_per_game, 2),
#             "Home_xGA_per_game": round(home_xga_per_game, 2),
#             "Away_xG_per_game": round(away_xg_per_game, 2),
#             "Away_xGA_per_game": round(away_xga_per_game, 2),
#         }
#     )

# df = pd.DataFrame(league_data)

# print("\nV3.0 Time-Decayed Data Extraction Complete.")
# display(df.head())

from util import build_regression_log

long_df = build_regression_log(extracted_teams, target_date)

print(long_df.head())

Stitching schedule and building Long-Format Data...
Successfully stitched 522 match records.
               Team          Opponent Venue        xG    Weight
0         Liverpool       Bournemouth  Home  2.330070  0.219320
1       Bournemouth         Liverpool  Away  1.573030  0.219320
2       Aston Villa  Newcastle United  Home  0.318601  0.221016
3  Newcastle United       Aston Villa  Away  1.400980  0.221016
4        Sunderland          West Ham  Home  0.724368  0.221016


In [ ]:
# Calculate ALL 4 League Averages using the decayed data
league_avg_home_xg = df["Home_xG_per_game"].mean()
league_avg_away_xga = df["Away_xGA_per_game"].mean()

league_avg_away_xg = df["Away_xG_per_game"].mean()
league_avg_home_xga = df["Home_xGA_per_game"].mean()

# Calculate the 4 Core Strengths
df["Home_Attack"] = df["Home_xG_per_game"] / league_avg_away_xga
df["Away_Attack"] = df["Away_xG_per_game"] / league_avg_home_xga
df["Home_Defense"] = df["Home_xGA_per_game"] / league_avg_away_xg
df["Away_Defense"] = df["Away_xGA_per_game"] / league_avg_home_xg

print("\nV3.0 Model Parameters Calculated (Time Decay Active):")
display(
    df[["Team", "Home_Attack", "Home_Defense", "Away_Attack", "Away_Defense"]].head(10)
)


V3.0 Model Parameters Calculated (Time Decay Active):


,Team,Home_Attack,Home_Defense,Away_Attack,Away_Defense
0,Aston Villa,0.795587,0.667614,1.066098,1.160350
1,Everton,0.714286,1.093750,0.902630,0.950437
2,Bournemouth,1.039489,0.625000,1.165601,1.282799
3,Sunderland,0.754936,1.235795,0.568586,1.113703
4,Crystal Palace,1.114983,1.129261,0.938166,0.857143
5,Chelsea,1.393728,1.100852,1.691542,0.892128
6,West Ham,0.993031,0.823864,0.959488,1.329446
7,Tottenham,0.737515,1.448864,0.810235,0.909621
8,Arsenal,1.103368,0.539773,1.577825,0.419825
9,Newcastle United,1.457607,1.079545,0.803127,0.956268


In [ ]:
from util import calculate_match_lambdas_v2

# --- Test it out ---
# Make sure 'league_avg_xg_per_game' is defined from your previous cell
home_team_test = "Burnley"  # Replace with a valid name from your 'Team' column
away_team_test = "Liverpool"

home_xg, away_xg = calculate_match_lambdas_v2(
    home_team_test, away_team_test, df, league_avg_home_xg, league_avg_away_xg
)

print(
    f"Projected Matchup: {home_team_test} ({home_xg} xG) vs {away_team_test} ({away_xg} xG)"
)

Projected Matchup: Burnley (0.981 xG) vs Liverpool (2.459 xG)


In [ ]:
from util import calculate_match_probabilities

# --- Test the Dixon-Coles Engine ---
# Let's test it with a tightly contested match where draws are highly likely
home_xg_test = 1.2
away_xg_test = 1.1

# Run without Dixon-Coles (Pure Poisson)
pure_poisson = calculate_match_probabilities(home_xg_test, away_xg_test, rho=0.0)

# Run WITH Dixon-Coles (rho = -0.15)
dc_adjusted = calculate_match_probabilities(home_xg_test, away_xg_test, rho=-0.15)

print("--- The Dixon-Coles Impact (Draw Probabilities) ---")
print(f"Pure Poisson 0-0: {pure_poisson['Matrix'].iloc[0,0] * 100:.2f}%")
print(f"DC Adjusted 0-0:  {dc_adjusted['Matrix'].iloc[0,0] * 100:.2f}%\n")

print(f"Pure Poisson Draw Total: {pure_poisson['Draw_Prob'] * 100:.2f}%")
print(f"DC Adjusted Draw Total:  {dc_adjusted['Draw_Prob'] * 100:.2f}%")

--- The Dixon-Coles Impact (Draw Probabilities) ---
Pure Poisson 0-0: 10.03%
DC Adjusted 0-0:  12.01%

Pure Poisson Draw Total: 28.32%
DC Adjusted Draw Total:  32.29%


In [ ]:
import pandas as pd
from util import find_value_bets


# --- Test the Arbitrage Scanner ---
# Let's pretend Pinnacle has these odds for the match:
# Home: 2.10, Draw: 3.50, Away: 3.20
pinnacle_odds = {"Home": 2.10, "Draw": 3.50, "Away": 3.20}

# Pass in the 'match_results' dictionary we generated in Step 3
value_dashboard = find_value_bets(dc_adjusted, pinnacle_odds)

print("--- Market Analysis Dashboard ---")
display(value_dashboard)

--- Market Analysis Dashboard ---


,Market,Model_Prob,Model_Odds,Bookie_Odds,Edge (EV),Bet_Signal
0,Home,36.12%,2.77,2.1,-24.16%,PASS
1,Draw,32.29%,3.10,3.5,13.03%,🔥 VALUE (BET)
2,Away,31.53%,3.17,3.2,0.91%,🔥 VALUE (BET)
